# Hlavní Prework Projekt: Příprava dat pro model ve strojovém učení (Cancer Dataset)

Tento sešit představuje kompletní praktickou implementaci kroků předzpracování a přípravy dat (**Data Preparation Pipeline**) před samotným trénováním modelů strojového učení.

### Klíčové fáze pipeline:
1. **Načtení a prvotní inspekce dat** (`pd.read_csv`, `.head()`, `.info()`, `.isnull()`)
2. **Odstranění redundantních proměnných** (`id`, `Unnamed: 32`)
3. **Zpracování chybějících hodnot** (filtrace chybějících řádků přes `.dropna()` vs. imputace přes `SimpleImputer`)
4. **Detekce a odstranění duplicit** (`.duplicated()`)
5. **Kódování kategorických proměnných** (normalizace velikosti písmen, `pd.get_dummies()`, `OneHotEncoder`)
6. **Analýza numerických vlastností a vizualizace** (`.describe()`, korelační heatmapa, rozdělení proměnných)
7. **Škálování a normalizace příznaků** (`MinMaxScaler` ze Scikit-learn vs. vlastní funkce)
8. **Rozdělení dat na podmnožiny** (trénovací, validační a testovací sada pomocí `train_test_split`)

## 1. Načtení datasetu a prvotní inspekce

Dataset obsahuje parametry diagnostiky nádorových buněk (např. průměrný poloměr `radius_mean`, texturu `texture_mean`, hladkost `smoothness_mean`, atd.). Cílem je předpovědět, zda je nádor nezhoubný (**B** - Benign) nebo zhoubný (**M** - Malignant).

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Načtení datasetu
data_path = os.path.join("data", "cancer_data_course.csv")
cancer_df = pd.read_csv(data_path)

print(f"Rozměry datasetu: {cancer_df.shape[0]} řádků, {cancer_df.shape[1]} sloupců")
cancer_df.head(10)

In [ ]:
# Zobrazení datových typů a nenulových hodnot
cancer_df.info()

## 2. Odstranění redundantních proměnných

Sloupec `id` je pouze ordinální identifikátor vyšetření a nenese žádnou prediktivní hodnotu pro klasifikaci. Sloupec `Unnamed: 32` (a případně exportní `Unnamed: 0`) vznikl chybou při exportu a obsahuje výhradně prázdné hodnoty.

In [ ]:
# Identifikace a odstranění zbytečných sloupců
redundant_cols = [c for c in ["id", "Unnamed: 32", "Unnamed: 0"] if c in cancer_df.columns]
cancer_df = cancer_df.drop(columns=redundant_cols)

print(f"Odstraněno: {redundant_cols}")
print(f"Nový tvar tabulky: {cancer_df.shape}")

## 3. Zpracování chybějících hodnot (Missing Values)

Nejprve spočítáme procentuální zastoupení chybějících hodnot v jednotlivých sloupcích.

In [ ]:
# Procentuální podíl chybějících hodnot ve sloupcích
missing_pct = (cancer_df.isnull().sum() / len(cancer_df)) * 100
missing_summary = pd.DataFrame({
    "Chybějící hodnoty": cancer_df.isnull().sum(),
    "Podíl (%)": missing_pct.round(2)
})
missing_summary[missing_summary["Chybějící hodnoty"] > 0]

### Varianta A: Odstranění řádků s chybějícími hodnotami (`dropna`)
Protože je podíl chybějících hodnot velmi malý (kolem 0.2 - 1.2 %), můžeme tyto řádky bezpečně zahodit. Po tomto kroku nám zůstane přesně **549 řádků**.

In [ ]:
# Odstranění řádků s alespoň jednou chybějící hodnotou a reset indexu
cancer_df = cancer_df.dropna(axis=0).reset_index(drop=True)
print(f"Počet řádků po dropna: {len(cancer_df)}")

### Varianta B (Alternativa): Imputace pomocí `SimpleImputer` ze Scikit-learn
Pokud bychom řádky nechtěli mazat, můžeme prázdná místa doplnit průměrem (`mean`), mediánem (`median`) či nejčastější hodnotou.

In [ ]:
from sklearn.impute import SimpleImputer

# Ukázka imputace průměrem pro numerické sloupce
imputer = SimpleImputer(strategy="mean")
numeric_cols = cancer_df.select_dtypes(include=[np.number]).columns
imputed_array = imputer.fit_transform(cancer_df[numeric_cols])
print(f"Imputováno {imputed_array.shape[1]} numerických vlastností.")

## 4. Detekce a odstranění duplicit

Metodou `.duplicated().sum()` ověříme přítomnost duplicitních řádků.

In [ ]:
num_duplicates = cancer_df.duplicated().sum()
print(f"Počet duplicitních řádků v datasetu: {num_duplicates}")

if num_duplicates > 0:
    cancer_df = cancer_df.drop_duplicates().reset_index(drop=True)

## 5. Kódování kategorických proměnných (`diagnosis`)

Zkontrolujeme hodnoty v proměnné `diagnosis`. Najdeme zde hodnoty `B` (Benign), `M` (Malignant) a také malá písmena `b` vzniklá překlepem při zadávání dat.

In [ ]:
print("Počet unikátních hodnot:", cancer_df["diagnosis"].nunique())
print("\nČetnosti jednotlivých hodnot před opravou:")
print(cancer_df["diagnosis"].value_counts())

# Normalizace překlepu: převod 'b' na 'B'
cancer_df["diagnosis"] = cancer_df["diagnosis"].replace("b", "B")

print("\nČetnosti po opravě:")
print(cancer_df["diagnosis"].value_counts())

### Kódování pomocí `pd.get_dummies(..., drop_first=True)` [Doporučeno]

Použití `drop_first=True` vytvoří jediný sloupec `diagnosis_M`:
- `1` = Malignant (zhoubný nádor)
- `0` = Benign (nezhoubný nádor)

In [ ]:
cancer_encoded_df = pd.get_dummies(cancer_df, columns=["diagnosis"], drop_first=True, dtype=int)
target_col = [c for c in cancer_encoded_df.columns if "diagnosis" in c][0]
print(f"Cílová zakódovaná proměnná: '{target_col}'")
cancer_encoded_df[[target_col]].value_counts()

## 6. Analýza vztahů v numerických datech a vizualizace

Prozkoumáme základní statistiky (průměr, rozptyl, minima, maxima), abychom odhalili rozdíly v měřítkách jednotlivých proměnných.

In [ ]:
# Přehled základních statistik
cancer_encoded_df.describe().T[["mean", "std", "min", "50%", "max"]].head(10)

### Korelační matice a vizualizace pomocí Seaborn heatmapy

In [ ]:
corr_matrix = cancer_encoded_df.corr()

plt.figure(figsize=(16, 14))
sns.heatmap(corr_matrix, cmap="coolwarm", annot=False, linewidths=0.5)
plt.title("Korelační matice proměnných v Cancer Datasetu", fontsize=15, fontweight="bold", pad=12)
plt.tight_layout()
plt.show()

### Rozdělení hodnot vybrané proměnné (`radius_mean`)
Hodnoty v mnoha sloupcích neodpovídají normálnímu Gaussovu rozdělení, proto pro sjednocení měřítek použijeme **Min-Max normalizaci** namísto standardizace (Z-score).

In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(cancer_encoded_df["radius_mean"], kde=True, color="#2b5c8f", bins=30)
plt.title("Distribuce proměnné 'radius_mean' (Histogram + KDE křivka)", fontsize=13, fontweight="bold")
plt.xlabel("radius_mean")
plt.ylabel("Počet pozorování")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Normalizace dat (MinMaxScaler)

Škálováním převedeme všechny numerické sloupce do jednotného rozsahu $[0, 1]$:
$$
x_{\text{norm}} = \frac{x - x_{\min}}{x_{\max} - x_{\min}}
$$

In [ ]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
cancer_normalized = scaler.fit_transform(cancer_encoded_df)
cancer_normalized_df = pd.DataFrame(data=cancer_normalized, columns=scaler.get_feature_names_out())

print("Normalizovaný dataset:")
cancer_normalized_df.head()

## 8. Rozdělení dat na trénovací, validační a testovací množinu

Nejprve oddělíme matici příznaků $X$ od vektoru cílové proměnné $y$ (`diagnosis_M`).

In [ ]:
X = cancer_normalized_df.drop(columns=[target_col])
y = cancer_normalized_df[target_col]

print(f"Tvar matice příznaků X: {X.shape}")
print(f"Tvar cílové proměnné y: {y.shape}")

### A) 2-cestné rozdělení (70 % trénovací / 30 % testovací sada)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42
)

print("2-cestný split:")
print(f"- X_train: {X_train.shape} ({len(X_train)/len(X)*100:.1f} %)")
print(f"- X_test:  {X_test.shape} ({len(X_test)/len(X)*100:.1f} %)")

### B) 3-cestné rozdělení (70 % trénovací / 15 % validační / 15 % testovací sada)

Provedeme dodatečné rozdělení testovací sady v poměru 50:50 na validační a finální testovací sadu.

In [ ]:
X_test_final, X_valid, y_test_final, y_valid = train_test_split(
    X_test, y_test, test_size=0.50, random_state=42
)

print("3-cestný split:")
print(f"- X_train:      {X_train.shape} (70 %)")
print(f"- X_valid:      {X_valid.shape} (15 %)")
print(f"- X_test_final: {X_test_final.shape} (15 %)")

## Shrnutí
V tomto projektu jsme úspěšně realizovali celý proces přípravy dat:
- Odstranění zbytečných a prázdných sloupců.
- Ošetření chybějících hodnot a normalizaci textových chyb (`b` -> `B`).
- Zakódování cílové proměnné na binární tvar pomocí `pd.get_dummies`.
- Prozkoumání korelací a distribucí.
- Min-Max škálování do intervalu $[0, 1]$.
- Rozdělení do trénovací, validační a testovací sady připravené pro trénování modelů strojového učení.